# Notebook 04 — Bayesian Optimization for Tabular Regression

**Session:** Hyper-parameter optimization  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Understand how **Bayesian Optimization** optimizes expensive black-box validation
objectives by fitting a probabilistic **surrogate model** (Gaussian Process) and
maximizing an **acquisition function** (Expected Improvement) to balance *exploration*
and *exploitation*.

1. **Part A (Pure NumPy + SciPy, runs immediately offline):** Tune Ridge regression
   regularization $\lambda$ and polynomial/RBF kernel scale on the portable 500-row sample
   `../data/tabular_regression_sample.csv` using a Gaussian Process surrogate and
   Expected Improvement (EI) from scratch.
2. **Part B (Production AutoML Pattern):** Configure `skopt.BayesSearchCV` with
   `Real`/`Integer` log-uniform priors, `DeltaYStopper`, and `DeadlineStopper` for
   `lightgbm.LGBMRegressor`.

## Part A — Gaussian Process + Expected Improvement from Scratch

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import cho_factor, cho_solve
from scipy.stats import norm

In [ ]:
def load_tabular_sample(csv_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load tabular sample CSV, ordinal-encode categoricals, and return (X, y)."""
    with csv_path.open(encoding="utf-8") as fp:
        reader = csv.DictReader(fp)
        rows = list(reader)

    cat_cols = [f"cat{i}" for i in range(10)]
    cont_cols = [f"cont{i}" for i in range(14)]

    # Build deterministic category-to-integer vocabulary
    vocab: dict[str, int] = {}
    for row in rows:
        for col in cat_cols:
            val = row[col]
            if val not in vocab:
                vocab[val] = len(vocab)

    x_list = []
    y_list = []
    for row in rows:
        cats = [float(vocab[row[c]]) for c in cat_cols]
        conts = [float(row[c]) for c in cont_cols]
        x_list.append(cats + conts)
        y_list.append(float(row["target"]))

    return np.asarray(x_list, dtype=np.float64), np.asarray(y_list, dtype=np.float64)


X_raw, y_vec = load_tabular_sample(Path("../data/tabular_regression_sample.csv"))
# Standardize features for kernel Ridge regression
X_scaled = (X_raw - X_raw.mean(axis=0)) / (X_raw.std(axis=0) + 1e-8)
print(f"Loaded sample dataset: X={X_scaled.shape}, y={y_vec.shape}")

In [ ]:
def kfold_kernel_ridge_neg_rmse(
    features: np.ndarray,
    targets: np.ndarray,
    log10_alpha: float,
    log10_gamma: float,
    n_splits: int = 5,
) -> float:
    """Return negative 5-fold CV RMSE for RBF Kernel Ridge Regression."""
    alpha = 10.0**log10_alpha
    gamma = 10.0**log10_gamma
    n = len(features)
    rng = np.random.default_rng(0)
    indices = rng.permutation(n)
    folds = np.array_split(indices, n_splits)
    rmses = []

    for val_idx in folds:
        mask = np.ones(n, dtype=bool)
        mask[val_idx] = False
        x_tr, y_tr = features[mask], targets[mask]
        x_va, y_va = features[val_idx], targets[val_idx]

        # RBF kernel matrices
        d2_tr = np.sum((x_tr[:, None, :] - x_tr[None, :, :]) ** 2, axis=-1)
        k_tr = np.exp(-gamma * d2_tr)
        k_tr[np.diag_indices_from(k_tr)] += alpha

        weights = np.linalg.solve(k_tr, y_tr)

        d2_va = np.sum((x_va[:, None, :] - x_tr[None, :, :]) ** 2, axis=-1)
        k_va = np.exp(-gamma * d2_va)
        preds = k_va @ weights
        rmses.append(float(np.sqrt(np.mean((preds - y_va) ** 2))))

    return -float(np.mean(rmses))


def rbf_kernel(mat_a: np.ndarray, mat_b: np.ndarray, length_scale: float = 1.0) -> np.ndarray:
    """Compute isotropic Gaussian Process RBF covariance matrix."""
    sq_dist = np.sum((mat_a[:, None, :] - mat_b[None, :, :]) ** 2, axis=-1)
    return np.exp(-0.5 * sq_dist / (length_scale**2))


def expected_improvement(
    candidates: np.ndarray,
    x_obs: np.ndarray,
    y_obs: np.ndarray,
    length_scale: float = 1.2,
    noise: float = 1e-4,
    xi: float = 0.01,
) -> np.ndarray:
    """Compute GP posterior mean, std, and Expected Improvement over *candidates*."""
    k_mat = rbf_kernel(x_obs, x_obs, length_scale) + noise * np.eye(len(x_obs))
    chol_l, lower = cho_factor(k_mat, lower=True)
    alpha_vec = cho_solve((chol_l, lower), y_obs - np.mean(y_obs))

    k_star = rbf_kernel(candidates, x_obs, length_scale)
    mu = np.mean(y_obs) + k_star @ alpha_vec

    v = cho_solve((chol_l, lower), k_star.T)
    var = np.maximum(1.0 - np.sum(k_star.T * v, axis=0), 1e-12)
    sigma = np.sqrt(var)

    best_y = np.max(y_obs)
    imp = mu - best_y - xi
    z = imp / sigma
    ei = imp * norm.cdf(z) + sigma * norm.pdf(z)
    return ei

In [ ]:
# Candidate grid in (log10_alpha, log10_gamma) search space
alphas = np.linspace(-3.0, 2.0, 21)
gammas = np.linspace(-4.0, 0.0, 21)
grid = np.array([[a, g] for a in alphas for g in gammas])

# Seed with 4 initial random evaluations
rng = np.random.default_rng(42)
init_indices = rng.choice(len(grid), size=4, replace=False)
X_obs = grid[init_indices]
y_obs = np.array([kfold_kernel_ridge_neg_rmse(X_scaled, y_vec, a, g) for a, g in X_obs])

# Sequential Bayesian Optimization loop (10 iterations)
for step in range(10):
    ei_scores = expected_improvement(grid, X_obs, y_obs)
    next_idx = int(np.argmax(ei_scores))
    next_pt = grid[next_idx]
    next_score = kfold_kernel_ridge_neg_rmse(X_scaled, y_vec, next_pt[0], next_pt[1])
    X_obs = np.vstack([X_obs, next_pt])
    y_obs = np.append(y_obs, next_score)
    print(
        f"Iter {step + 1:2d}: log10_alpha={next_pt[0]:+.2f}, "
        f"log10_gamma={next_pt[1]:+.2f}, CV neg-RMSE={next_score:.4f}"
    )

best_idx = int(np.argmax(y_obs))
print(
    f"\nBest found: alpha={10 ** X_obs[best_idx, 0]:.4g}, "
    f"gamma={10 ** X_obs[best_idx, 1]:.4g}, best CV RMSE={-y_obs[best_idx]:.4f}"
)

plt.figure(figsize=(7, 4))
plt.plot(np.maximum.accumulate(y_obs), marker="o")
plt.xlabel("Evaluation Step")
plt.ylabel("Best Observed Neg-RMSE")
plt.title("Bayesian Optimization Convergence Trace")
plt.tight_layout()
plt.show()

## Part B — Production `BayesSearchCV` Pattern with LightGBM

When tuning tree ensembles like `lightgbm.LGBMRegressor` on the full dataset,
`skopt.BayesSearchCV` automates Gaussian Process surrogate search with log-uniform
search bounds and early-stopping callbacks (`DeltaYStopper` and `DeadlineStopper`).
The cell below runs automatically if `lightgbm`, `pandas`, and `scikit-optimize`
are installed in your environment.

In [ ]:
try:
    import pprint
    from functools import partial
    from time import time

    import lightgbm as lgb
    import pandas as pd
    from sklearn.metrics import make_scorer, root_mean_squared_error
    from sklearn.model_selection import KFold
    from skopt import BayesSearchCV
    from skopt.callbacks import DeadlineStopper, DeltaYStopper
    from skopt.space import Integer, Real

    HAS_SKOPT_LGB = True
except ImportError as exc:
    HAS_SKOPT_LGB = False
    print(f"Optional packages (lightgbm / skopt / pandas) not installed: {exc}")

if HAS_SKOPT_LGB:
    df = pd.read_csv("../data/tabular_regression_sample.csv")
    y_full = df["target"]
    X_full = df.set_index("id").drop("target", axis="columns")

    categoricals = [col for col in X_full.columns if "cat" in col]
    cat_values = np.unique(X_full[categoricals].values)
    cat_dict = dict(zip(cat_values, range(len(cat_values)), strict=True))
    X_full[categoricals] = X_full[categoricals].replace(cat_dict).astype("category")

    scoring = make_scorer(partial(root_mean_squared_error), greater_is_better=False)
    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    reg = lgb.LGBMRegressor(
        boosting_type="gbdt",
        metric="rmse",
        objective="regression",
        n_jobs=1,
        verbose=-1,
        random_state=0,
    )

    search_spaces = {
        "learning_rate": Real(0.01, 1.0, "log-uniform"),
        "n_estimators": Integer(30, 500),
        "num_leaves": Integer(2, 128),
        "max_depth": Integer(-1, 64),
        "min_child_samples": Integer(1, 64),
        "subsample": Real(0.1, 1.0, "uniform"),
        "colsample_bytree": Real(0.1, 1.0, "uniform"),
        "reg_lambda": Real(1e-9, 100.0, "log-uniform"),
        "reg_alpha": Real(1e-9, 100.0, "log-uniform"),
    }

    opt = BayesSearchCV(
        estimator=reg,
        search_spaces=search_spaces,
        scoring=scoring,
        cv=kf,
        n_iter=15,
        n_points=1,
        n_jobs=-1,
        return_train_score=False,
        refit=False,
        optimizer_kwargs={"base_estimator": "GP"},
        random_state=0,
    )

    t0 = time()
    opt.fit(
        X_full,
        y_full,
        callback=[DeltaYStopper(delta=0.0001), DeadlineStopper(total_time=300)],
    )
    print(f"LGBM BayesSearchCV finished in {time() - t0:.2f}s, best score: {opt.best_score_:.4f}")
    pprint.pprint(opt.best_params_)